In [1]:
import yfinance as yf
import pandas as pd

ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
result = ticker_obj.get_earnings_dates(limit=25)  # default limit is often 10 — bump it up

print(result)
print(len(result))

                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2026-10-29 16:00:00-04:00          1.95           NaN          NaN
2026-07-30 16:00:00-04:00          1.83          5.75       215.02
2026-04-29 16:00:00-04:00          1.65          2.78        68.18
2026-02-05 16:00:00-05:00          1.95          1.95         0.22
2025-10-30 16:00:00-04:00          1.56          1.95        25.20
2025-07-31 16:00:00-04:00          1.33          1.68        26.13
2025-05-01 16:00:00-04:00          1.36          1.59        17.08
2025-02-06 16:00:00-05:00          1.48          1.86        25.36
2024-10-31 16:00:00-04:00          1.14          1.43        25.17
2024-08-01 12:00:00-04:00          1.02          1.26        23.76
2024-04-30 16:00:00-04:00          0.83          0.98        17.67
2024-02-01 16:00:00-05:00          0.80          1.00        24.38
2023-10-26 16:00:00-04:00          0.58          0.94        6

In [2]:
prices = yf.download(ticker, start='2020-01-01', end='2026-08-21')['Close']

if isinstance(prices, pd.DataFrame):
    prices = prices.iloc[:, 0]

prices = prices.sort_index()

[*********************100%***********************]  1 of 1 completed


In [3]:
returns_df = pd.DataFrame({
    'day1_date': prices.index[:-2],
    'day1_close': prices.values[:-2],
    'day2_date': prices.index[1:-1],
    'day2_close': prices.values[1:-1],
    'day3_date': prices.index[2:],
    'day3_close': prices.values[2:],
})

returns_df['return_2day'] = returns_df['day3_close'] / returns_df['day1_close'] - 1
print(returns_df.head())

   day1_date  day1_close  day2_date  day2_close  day3_date  day3_close  \
0 2020-01-02   94.900497 2020-01-03   93.748497 2020-01-06   95.143997   
1 2020-01-03   93.748497 2020-01-06   95.143997 2020-01-07   95.343002   
2 2020-01-06   95.143997 2020-01-07   95.343002 2020-01-08   94.598503   
3 2020-01-07   95.343002 2020-01-08   94.598503 2020-01-09   95.052498   
4 2020-01-08   94.598503 2020-01-09   95.052498 2020-01-10   94.157997   

   return_2day  
0     0.002566  
1     0.017008  
2    -0.005733  
3    -0.003047  
4    -0.004657  


In [4]:
earnings = result.copy()
earnings = earnings.reset_index()  # earnings date becomes a column, check the column name
print(earnings.columns)

Index(['Earnings Date', 'EPS Estimate', 'Reported EPS', 'Surprise(%)'], dtype='str')


In [10]:
earnings = earnings.rename(columns={'Earnings Date': 'earnings_date', 'Surprise(%)': 'surprise_pct'})
earnings['earnings_date'] = pd.to_datetime(earnings['earnings_date']).dt.tz_localize(None)

# Drop the future entry with no reported EPS/surprise
earnings = earnings.dropna(subset=['surprise_pct'])

returns_df['day2_date'] = pd.to_datetime(returns_df['day2_date']).dt.tz_localize(None)

# Merge: for each earnings date, find the matching Day 2 in returns_df.
# Earnings announcements sometimes happen after market close, so Day 2
# may need to be the NEXT trading day after the earnings date rather than
# an exact match — check yours and adjust if you get few/no matches.
merged = pd.merge(earnings, returns_df, left_on='earnings_date', right_on='day2_date', how='inner')

print(f"Matched {len(merged)} earnings events to price windows")
returns_df['day2_date_shifted'] = returns_df['day2_date']
# Try merging earnings_date against day1_date instead, treating earnings_date as "Day 1"
merged = pd.merge(earnings, returns_df, left_on='earnings_date', right_on='day1_date', how='inner')

Matched 0 earnings events to price windows


In [11]:
positive_surprises = merged[merged['surprise_pct'] > 0]

median_return = positive_surprises['return_2day'].median()
print(f"Median 2-day return following positive earnings surprises: {median_return:.4f} ({median_return*100:.2f}%)")

# Correlation between surprise magnitude and 2-day return
correlation = merged[['surprise_pct', 'return_2day']].corr()
print(correlation)

Median 2-day return following positive earnings surprises: nan (nan%)
              surprise_pct  return_2day
surprise_pct           NaN          NaN
return_2day            NaN          NaN


In [12]:
print(correlation)

              surprise_pct  return_2day
surprise_pct           NaN          NaN
return_2day            NaN          NaN


In [14]:
median_return

nan